1a. Small perturbations might change the best first split of a tree, which would consequently end up changing other splits of the tree as well to follow the "most appropriate" sequence of splits based on the current data available.

1b. This is good if we're trying to build a model that does well on the current data. However, machine learning is about performing better on unseen data, which is why having a highly adaptable structure might be worrying. We might be overfitting our data. Instead, there are other techniques such as tree pruning to make sure we capture most of the "essential information" of the dataset instead of perfectly adapting to each exception that might arise.

1c. This splits the data into "rectangles." It's important to note that higher-dimensional data might still have overlapping regions when we project it into 2d (because now we're not being able to show how they differ from each other).

Summary of the text:
A decision tree is a supervised learning model for classification or regression. It predicts by applying a sequence of if/then rules from the root node down to a leaf. Each internal node splits the feature space. For example, in the article’s tree-species example, splits are made using tree diameter and height to separate Apple, Cherry, and Oak trees. Training a tree means deciding which feature to split on and at what threshold. The goal is to make the resulting groups as pure as possible.

The article mainly explains this using entropy:
H=−k∑​pk​logpk​

In [ ]:
import numpy as np
from sklearn import metrics, inspection, datasets
from matplotlib import pyplot as plt

x_data, y_data = datasets.make_moons(n_samples=500, noise=0.15)

plt.scatter(x_data[:, 0], x_data[:, 1], c=y_data)
plt.show()

In [ ]:
def leaf_gini_impurity(leaf_ys):
    if len(leaf_ys) == 0:
        return 0
    else:
        return 1 - (sum(leaf_ys==0)/len(leaf_ys))**2 - (sum(leaf_ys==1)/len(leaf_ys))**2

def weighted_gini_impurity(left_ys, right_ys):
    total = len(left_ys) + len(right_ys)
    left_leaf = leaf_gini_impurity(left_ys) * len(left_ys)/total
    right_leaf = leaf_gini_impurity(right_ys) * len(right_ys)/total
    return left_leaf + right_leaf 

# You only need to use this function directly to solve this question:
def gini_impurity(x_data, y_data, axis, threshold):
    # The axis argument should be 0 or 1
    # The threshold should be a real valued number
    left_mask = x_data[:, axis] <= threshold
    right_mask = x_data[:, axis] > threshold
    return weighted_gini_impurity(y_data[left_mask], y_data[right_mask]) 

# Here's a sample way to use this function
best_gini = float("inf")
best_axis = None
best_threshold = None

# x is the dummy variable choosing the feature/axis: 0 or 1
for axis in [0, 1]:
    # y is the threshold, discretized by observed feature values
    thresholds = np.unique(x_data[:, axis])

    for threshold in thresholds:
        gini = gini_impurity(x_data,y_data,axis=axis,threshold=threshold)

        if gini < best_gini:
            best_gini = gini
            best_axis = axis
            best_threshold = threshold

print("Global minimum Gini impurity:")
print(f"Axis: {best_axis}")
print(f"Threshold: {best_threshold:.4f}")
print(f"Gini impurity: {best_gini:.4f}")

plt.scatter(x_data[:, 0], x_data[:, 1], c=y_data)

if best_axis == 0:
    # Split based on x-coordinate -> vertical line
    plt.axvline(best_threshold, linestyle="--")
else:
    # Split based on y-coordinate -> horizontal line
    plt.axhline(best_threshold, linestyle="--")

plt.xlabel("x")
plt.ylabel("y")
plt.show()

In [ ]:
# Helper function: find the best split for any subset of the data
def find_best_split(x_subset, y_subset):
    best_gini = float("inf")
    best_axis = None
    best_threshold = None

    for axis in [0, 1]:
        thresholds = np.unique(x_subset[:, axis])

        for threshold in thresholds:
            gini = gini_impurity(x_subset, y_subset, axis=axis, threshold=threshold)

            if gini < best_gini:
                best_gini = gini
                best_axis = axis
                best_threshold = threshold

    return best_axis, best_threshold, best_gini

# Divide the original data using the first split
left_mask = x_data[:, best_axis] <= best_threshold
right_mask = x_data[:, best_axis] > best_threshold

x_left = x_data[left_mask]
y_left = y_data[left_mask]

x_right = x_data[right_mask]
y_right = y_data[right_mask]

# Find the best split in the LEFT half
left_axis, left_threshold, left_gini = find_best_split(x_left, y_left)

# Find the best split in the RIGHT half
right_axis, right_threshold, right_gini = find_best_split(x_right, y_right)

# Results
axis_names = {0: "x", 1: "y"}

print("First Split")
print(f"Axis: {axis_names[best_axis]} ({best_axis})")
print(f"Threshold: {best_threshold:.4f}")
print(f"Gini: {best_gini:.4f}")

print("\nSecond Split - Left Child")
print(f"Axis: {axis_names[left_axis]} ({left_axis})")
print(f"Threshold: {left_threshold:.4f}")
print(f"Gini: {left_gini:.4f}")

print("\nSecond Split - Right Child")
print(f"Axis: {axis_names[right_axis]} ({right_axis})")
print(f"Threshold: {right_threshold:.4f}")
print(f"Gini: {right_gini:.4f}")